# Generate stationary and dynamic textures for experiment


In [1]:
from dyntex.MotionCloud import MotionCloud
from dyntex.DriftingGrating import DriftingGrating

import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation
import matplotlib as mpl

import torch as tch

import imageio as imio
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import i0, i1
import time
import shutil
import imageio.v2 as imageio

import os
import re


In [2]:
mpl.rcParams['animation.writer'] = 'pillow'
mpl.rcParams['animation.embed_limit'] = 1000

if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg not found on PATH. Install or add it to PATH.')


## Utility functions


In [3]:
def to_numpy(x):
    return x.detach().cpu().numpy()


def write_mp4_safe(mp4_path, frames, fps=30, codec='libx264'):
    frames = np.asarray(frames)
    h, w = frames.shape[1], frames.shape[2]

    def prepare_frame(f):
        f = np.clip(f, 0, 255).astype(np.uint8)
        if f.ndim == 2:
            f = np.stack((f,) * 3, axis=-1)
        elif f.shape[2] == 1:
            f = np.concatenate((f,) * 3, axis=2)
        return f

    try:
        with imageio.get_writer(mp4_path, fps=fps, codec=codec, format='FFMPEG') as writer:
            for f in frames:
                writer.append_data(prepare_frame(f))
    except Exception:
        with imageio.get_writer(mp4_path, fps=fps, codec=codec, format='ffmpeg') as writer:
            for f in frames:
                writer.append_data(prepare_frame(f))


## Temporal profile generation


In [4]:
def linear_feature_profile_speed_per_frame(
    num_frames,
    start_frame,
    start_value,
    speed_per_frame,
    variation_duration_frames
):
    num_frames = int(num_frames)
    start_frame = int(start_frame)
    variation_duration_frames = int(variation_duration_frames)
    if variation_duration_frames <= 0:
        raise ValueError('variation_duration_frames must be >0')
    if start_frame < 0:
        raise ValueError('start_frame must be >= 0')
    if start_frame + variation_duration_frames > num_frames:
        raise ValueError('Ramp exceeds total number of frames')

    feature_value = np.full(num_frames, float(start_value), dtype=float)

    # rampe linéaire
    t = np.arange(variation_duration_frames, dtype=float)
    ramp = float(start_value) + float(speed_per_frame) * t

    feature_value[start_frame:start_frame + variation_duration_frames] = ramp

    # valeur stationnaire finale
    final_value = ramp[-1]
    feature_value[start_frame + variation_duration_frames:] = final_value

    return feature_value


## Stimulus generation


Adding Cesar's dynamic code below

In [5]:
def linear_feature_profile_speed_per_frame(
    num_frames,
    start_frame,
    start_value,
    speed_per_frame,
    variation_duration_frames
):
    num_frames = int(num_frames)
    start_frame = int(start_frame)
    variation_duration_frames =int(variation_duration_frames)
    if variation_duration_frames<=0:
        raise ValueError("variation_duration_frames must be >0")
    if start_frame < 0:
        raise ValueError("start_frame must be >= 0")
    if start_frame + variation_duration_frames > num_frames:
        raise ValueError("Ramp exceeds total number of frames")

    feature_value = np.full(num_frames, float(start_value), dtype=float)

    # rampe linéaire
    t = np.arange(variation_duration_frames, dtype=float)
    ramp = float(start_value) + float(speed_per_frame)* t

    feature_value[start_frame : start_frame + variation_duration_frames] = ramp

    # valeur stationnaire finale
    final_value = ramp[-1]
    feature_value[start_frame + variation_duration_frames:] = final_value

    return feature_value

In [6]:
def generate_nonstationary_MC_second_unit(
    stimulus_duration_s,
    feature_name,
    feature_start_value,
    feature_speed,
    variation_duration,
    save_npz=False,
    npz_path="stimulus.npz",
    save_mp4=False,
    mp4_path="stimulus.mp4"
):

    params = {
        "sf": 5.0,
        "sf_sig": 1.0, # sf bandwidth, to change
        "th": 0.0,
        "th_sig": 5.0, # th bandwidth
        "tf": 5.0,
        "speed_vec": tch.tensor([0.0, 0.0]),
        "speed_sig": 140.0,
        "octa": 1,
        "N": 256,
        "frame_per_second": 100,
        "device": "cuda" if tch.cuda.is_available() else "cpu"
    }

    fps_native = int(params["frame_per_second"])

    # Durée totale -> nombre de frames
    num_frames = int(np.round(float(stimulus_duration_s) * fps_native))

    # variation_duration reste en secondes
    variation_duration_frames = int(np.round(float(variation_duration) * fps_native))
    if variation_duration_frames <= 0:
        raise ValueError("variation_duration (in seconds) is too small")
    if variation_duration_frames > num_frames:
        raise ValueError("variation_duration is longer than stimulus")

    speed_per_frame = float(feature_speed) / float(fps_native)

    # Début du changement fixé à 2650 ms
    start_time = 3000
    start_frame = int(np.round(start_time * fps_native / 1000.0))

   # time_max : moment à partir duquel il reste encore 3130 ms
    total_duration_ms = float(stimulus_duration_s) * 1000.0
    time_max_ms = total_duration_ms - 3130.0

    if time_max_ms < 0:
        raise ValueError("stimulus_duration_s is shorter than 3130 ms, so time_max_ms is negative")

    # Frame associée à time_max_ms
    time_max_frame = int(np.round(time_max_ms * fps_native / 1000.0))
    time_max_frame = int(np.clip(time_max_frame, 0, num_frames - 1))


    # Profil temporel
    L = linear_feature_profile_speed_per_frame(
        num_frames=num_frames,
        start_frame=start_frame,
        start_value=feature_start_value,
        speed_per_frame=speed_per_frame,
        variation_duration_frames=variation_duration_frames
    )

    print(L)
    feature_end_value = L[start_frame + variation_duration_frames - 1]
    print(feature_end_value)
    
    MC = MotionCloud(
        N=params["N"],
        frame_per_second=fps_native,
        contrast=35,
        ave_lum=127.0,
        over_samp=10,
        verbose=0
    )
    MC.set_grids()

    frames = tch.zeros(
        (num_frames, params["N"], params["N"]),
        dtype=tch.float32,
        device=params["device"]
    )

    spd_scalar = 0.0
    spd_dir = 0.0

    for i, value in enumerate(L):
        if feature_name == "sf":
            sf = float(value)
            th = float(params["th"])
        elif feature_name == "th":
            sf = float(params["sf"])
            th = float(value)
        else:
            raise ValueError("feature_name must be 'sf' or 'th'")

        MC.set_parameters(
            sf=sf,
            sf_sig=float(params["sf_sig"]),
            th=th,
            th_sig=float(params["th_sig"]),
            tf=float(params["tf"]),
            spd_scalar=spd_scalar,
            spd_dir=spd_dir,
            octa=int(params["octa"])
        )
        MC.set_ar_coeffs()
        MC.bandpass_kernel()

        if i == 0:
            MC.set_fourier_translation()

        frame = MC.get_frame(adjust=True)
        frames[i] = frame

    if isinstance(frames, tch.Tensor):
        frames = frames.detach().cpu().numpy()

    if save_npz:
        np.savez(
            npz_path,
            frames=frames,
            feature_name=feature_name,
            profile=L,
            time_max_frame=time_max_frame,
            fps=fps_native,
            feature_start_value = feature_start_value,
            feature_end_value = feature_end_value 
        )

    if save_mp4:
        write_mp4_safe(mp4_path, frames)
    
    print(time_max_frame)

    return frames, time_max_ms, time_max_frame

Generated some missing, modify this function as needed or comment out to avoid generating tons of files

In [ ]:
# for speed in np.arange(16, 56, 2.0):  
#     speed_rounded = round(-speed, 2) 
#     A = generate_nonstationary_MC_second_unit(
#         stimulus_duration_s=4.0,
#         feature_name="th",
#         feature_start_value=0.0,
#         feature_speed=speed_rounded,
#         variation_duration=0.2,
#         save_npz=True,
#         npz_path=f"dynamic_stimulus_orientation_{speed_rounded}.npz",
#         save_mp4=False,
#         mp4_path=f"stimulus_orientation_{speed_rounded}.mp4"
#     )

Block to print all missing from experiment directory

In [ ]:
# directory that already contains some finished npz files
existing_dir = "/Users/maggiebowen/Documents/Dugue-Research/classif-waves-experiment/new-stim/changing-stim/dynamic_stimulus_orientation"

# output stays in current working directory
output_dir = os.getcwd()

# robust filename parser
pattern = re.compile(r"^dynamic_stimulus_orientation_(-?\d+(?:\.\d+)?)\.npz$")

def collect_speeds(folder):
    speeds = set()
    for fname in os.listdir(folder):
        m = pattern.match(fname)
        if m:
            speeds.add(int(round(float(m.group(1)))))
    return speeds

# check both:
# 1. external directory with old files
# 2. working directory in case some were already generated here too
existing_external = collect_speeds(existing_dir)
existing_local = collect_speeds(output_dir)

existing_speeds = existing_external | existing_local

# full desired range: every integer speed from -140 to 140 inclusive
target_speeds = list(range(-140, 141))

missing_speeds = [s for s in target_speeds if s not in existing_speeds]

print(f"Checked existing directory: {existing_dir}")
print(f"Checked working directory: {output_dir}")
print(f"Found {len(existing_external)} existing in external directory")
print(f"Found {len(existing_local)} existing in working directory")
print(f"Found {len(existing_speeds)} unique existing total")
print(f"Missing {len(missing_speeds)} files")
print(missing_speeds)

for speed in missing_speeds:
    speed_rounded = round(float(speed), 2)
    print(f"Generating {speed_rounded}")

    A = generate_nonstationary_MC_second_unit(
        stimulus_duration_s=4.0,
        feature_name="th",
        feature_start_value=0.0,
        feature_speed=speed_rounded,
        variation_duration=0.2,
        save_npz=True,
        npz_path=f"dynamic_stimulus_orientation_{speed_rounded}.npz",
        save_mp4=False,
        mp4_path=f"stimulus_orientation_{speed_rounded}.mp4"
    )

Checked existing directory: /Users/maggiebowen/Documents/Dugue-Research/classif-waves-experiment/new-stim/changing-stim/dynamic_stimulus_orientation
Checked working directory: /Users/maggiebowen/Documents/Dugue-Research/dyntex_python/demo
Found 75 existing in external directory
Found 20 existing in working directory
Found 75 unique existing total
Missing 206 files
[-140, -139, -138, -137, -136, -135, -134, -133, -132, -131, -130, -129, -128, -127, -126, -125, -124, -123, -122, -121, -120, -119, -118, -117, -116, -115, -114, -113, -112, -111, -110, -109, -108, -107, -106, -105, -104, -103, -102, -101, -100, -99, -98, -97, -96, -95, -94, -93, -92, -91, -90, -89, -88, -87, -86, -85, -84, -83, -82, -81, -80, -79, -78, -77, -76, -75, -74, -73, -72, -71, -70, -69, -68, -67, -66, -65, -64, -63, -62, -61, -60, -59, -58, -57, -56, -55, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 40, 41, 42